Importation des packages a mettre dans requirements apres 

In [1]:
import geopandas as gpd
import pandas as pd

Importation des deux bases de données :
- Demandes de valeurs foncières
- Topologie des points d'arrêt de métro du réseau STAR

Points d'arrêt de métro 

In [2]:
# URL directe vers l'export GeoJSON du portail Open Data
url_metro = "https://data.rennesmetropole.fr/explore/dataset/topologie-des-points-darret-de-metro-du-reseau-star/download/?format=geojson"

# Lecture directe depuis l'URL
gdf_metro = gpd.read_file(url_metro)

# Aperçu des premières lignes et de la projection (WGS84 - EPSG:4326)
print(gdf_metro.head())
print(gdf_metro.crs)

  codeinseecommune                            coordonnees  \
0            35281  [48.0922543436006, -1.70335007013448]   
1            35238   [48.105036267316, -1.69241228789686]   
2            35238  [48.1042189200449, -1.67189613885799]   
3            35051  [48.1314259119796, -1.62024738680177]   
4            35051  [48.1271194329088, -1.62826328652191]   

  stop_id_station_parente adressenumero               adressevoie  \
0                 6-15064           1 B         rue Louis Braille   
1                 6-15061            15               rue Surcouf   
2                 6-15008            15          place de la Gare   
3                 6-15050            14               la Rochelle   
4                 6-15051             1  avenue de Belle Fontaine   

  estaccessiblepmr      nomstationparente stop_id    id  \
0             true  Saint-Jacques - Gaîté  6-5065  5065   
1             true               Mabilais  6-5068  5068   
2             true                  Gares

In [ ]:
print(gdf_metro.columns)


Index(['codeinseecommune', 'coordonnees', 'stop_id_station_parente',
       'adressenumero', 'adressevoie', 'estaccessiblepmr', 'nomstationparente',
       'stop_id', 'id', 'nomcommune', 'code', 'nom', 'idstationparente',
       'geometry'],
      dtype='object')


Demandes de Valeurs Foncières (DVF)

In [3]:
# Exemple pour l'année 2023 (ou la plus récente disponible sur l'API DVF)
# URL des fichiers consolidés par Etalab
url_dvf = "https://files.data.gouv.fr/geo-dvf/latest/csv/2023/full.csv.gz"

# On précise le séparateur (souvent la virgule) et on gère la compression .gz
# précisez 'low_memory=False' pour éviter les warnings sur les types
df_dvf = pd.read_csv(url_dvf, compression='gzip', sep=',', low_memory=False)

# Pour filtrer sur Rennes (code commune 35238)
df_rennes = df_dvf[df_dvf['code_commune'] == '35238'].copy()

# Aperçu des premières lignes et de la projection (WGS84 - EPSG:4326)
print(df_rennes.head())

         id_mutation date_mutation  numero_disposition nature_mutation  \
1365203  2023-471776    2023-01-04                   1           Vente   
1365204  2023-471776    2023-01-04                   1           Vente   
1365205  2023-471777    2023-01-07                   1           Vente   
1365206  2023-471777    2023-01-07                   1           Vente   
1365207  2023-471777    2023-01-07                   1           Vente   

         valeur_fonciere  adresse_numero adresse_suffixe adresse_nom_voie  \
1365203         160000.0            15.0             NaN       BD D ANJOU   
1365204         160000.0            15.0             NaN       BD D ANJOU   
1365205         320000.0           173.0             NaN  RUE DE FOUGERES   
1365206         320000.0           173.0             NaN  RUE DE FOUGERES   
1365207         320000.0           173.0             NaN  RUE DE FOUGERES   

        adresse_code_voie  code_postal  ...   type_local surface_reelle_bati  \
1365203     

In [4]:
print(df_rennes.columns)

Index(['id_mutation', 'date_mutation', 'numero_disposition', 'nature_mutation',
       'valeur_fonciere', 'adresse_numero', 'adresse_suffixe',
       'adresse_nom_voie', 'adresse_code_voie', 'code_postal', 'code_commune',
       'nom_commune', 'code_departement', 'ancien_code_commune',
       'ancien_nom_commune', 'id_parcelle', 'ancien_id_parcelle',
       'numero_volume', 'lot1_numero', 'lot1_surface_carrez', 'lot2_numero',
       'lot2_surface_carrez', 'lot3_numero', 'lot3_surface_carrez',
       'lot4_numero', 'lot4_surface_carrez', 'lot5_numero',
       'lot5_surface_carrez', 'nombre_lots', 'code_type_local', 'type_local',
       'surface_reelle_bati', 'nombre_pieces_principales',
       'code_nature_culture', 'nature_culture', 'code_nature_culture_speciale',
       'nature_culture_speciale', 'surface_terrain', 'longitude', 'latitude'],
      dtype='object')


Modèle de prédiction du prix :

In [8]:
from src.model import preparer_et_entrainer

# Supposons que df_rennes et gdf_metro sont déjà chargés
model_final, features_utilisees = preparer_et_entrainer(df_rennes, gdf_metro)

print("Modèle entraîné avec succès !")
print(f"Features utilisées : {features_utilisees}")

# Vous pouvez maintenant utiliser model_final pour faire des prédictions
# exemple_data = ... 
# prediction = model_final.predict(exemple_data)

Calcul des distances (cela peut prendre un instant)...


/opt/python/lib/python3.13/site-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


Modèle entraîné avec succès !
Features utilisées : ['surface_reelle_bati', 'nombre_pieces_principales', 'dist_min_metro', 'annee', 'type_local_code']


In [13]:

from src.model import preparer_et_entrainer, predire_impact_nouvelle_station

# Exemple : un appartement de 50m², 3 pièces, en 2026
surface = 50
pieces = 3
annee = 2026
code_type = 1 # Supposons que 1 soit l'appartement

# Scénario 1 : Le métro est à 2 km (2000m)
prix_actuel = predire_impact_nouvelle_station(model_final, surface, pieces, code_type, annee, distance_metro=2000)

# Scénario 2 : Le métro est à 0 m (nouvelle station en bas de l'immeuble)
prix_avec_metro = predire_impact_nouvelle_station(model_final, surface, pieces, code_type, annee, distance_metro=0)

print(f"Prix estimé actuel : {prix_actuel:.2f} €")
print(f"Prix estimé avec nouvelle station : {prix_avec_metro:.2f} €")
print(f"Plus-value théorique : {prix_avec_metro - prix_actuel:.2f} €")

ImportError: cannot import name 'predire_impact_nouvelle_station' from 'src.model' (/home/onyxia/work/Projet-python-pour-la-data-science/src/model.py)